# 5D Debug an Agent in VS Code

**Purpose:** Turn the small tool-using agent from [Section 4](../04_Agents/04_Agents_on_a_Governed_Endpoint.ipynb) into an ordinary Python module that you can test and debug before spending gateway tokens.

**What we're doing**

Run deterministic model responses, follow the agent through breakpoints, introduce and repair a token-accounting bug in a practice copy, and optionally make one bounded live run.

**Why we're doing it**

An agent loop needs tests for its stopping behavior as much as its happy path. The model decides what to request; your code controls which tools execute and whether another request is allowed.

**What to look for**

* The difference between a mocked response and a verified gateway capability
* Conversation history growing as tool results return
* Cumulative prompt/completion tokens, not just the most recent reply's usage
* A failed test becoming a useful debugger entry point

**Output:** Passing offline tests, a repaired practice copy, and an optional trace from the governed gateway.

> **Before you start:** Complete [the VS Code introduction](05_Running_VS_Code.ipynb). Open the repository root in the remote IDE. Use Python 3.10 or newer and the Microsoft Python and Python Debugger extensions supported by your IDE image (`ms-python.python`, `ms-python.debugpy`). Read Section 4's tools and agent loop. Offline work needs no model, GPU, token, `.env`, pip installation, or administrator setting changes.

This is a guided UI lab. Run fenced commands in the **remote IDE terminal**, not as notebook Python cells. The notebook has no stored execution output.

## 5D.1 Extract the Agent Without Carrying Notebook State

Open [`agent_lab.py`](agent_lab.py) and [`test_agent_lab.py`](test_agent_lab.py) side by side. The module adapts Section 4's `calculator`, `clock`, `lookup`, inventory, tool schema, and loop. Inventory is fictional teaching data, not live Klique pricing.

| Section 4 idea | Debuggable module |
|---|---|
| Globals for token and endpoint | Inject a `client`; load credentials only in the `--live` path |
| Native tool calls or JSON fallback | Explicit `mode`; JSON default sends no native `tools` |
| Trace and token tally | Return `trace`, `tokens`, and full `history` for inspection |
| Rate-limit retry loop | One attempt per step; stop on every gateway rejection |
| First JSON object found inside text | Strip `<think>` blocks, then require one complete JSON object |
| Tool functions | Read-only allowlist, argument checking, bounded arithmetic |

We deliberately **do not copy** Section 4's unknown-429 retries or its budget-exhaustion loop. A 429 can mean rate limiting or wallet exhaustion. This lab does not guess which and does not spend money trying to discover it. The test suite uses synthetic labeled errors; those labels are not evidence about your deployment.

The agent catches errors that expose `status_code` and `kind`, as the shared [`gateway_client.py`](gateway_client.py) does. Unexpected programming errors still propagate so the debugger can stop on them.

> **Checkpoint:** Importing `agent_lab` does not read secrets or contact the gateway.

## 5D.2 Establish a Passing Offline Baseline

In the repository-root terminal:

```bash
python --version
python 05_VSCode/agent_lab.py
python -m unittest discover -s 05_VSCode -p test_agent_lab.py -v
```

If this image uses `python3`, use that executable consistently. Select the same interpreter through **Python: Select Interpreter** in the Command Palette.

Expected demo output:

```text
step 1: lookup {"result": {"gpus": 8, "gpu_type": "A100 80GB", "hourly_rate_usd_per_gpu": 2.5}}
step 2: calculator {"result": 60.0}
step 3: - final answer
status: done; tokens: 45; answer: $60.00 for 3 hours
```

The tests should report **20 tests, OK**. The three mocked replies each contain 10 input and 5 output tokens, totaling 30 input + 15 output = 45. They are synthetic counts. The final wording is scripted, not proof of live model quality; the tests separately check the calculator result delivered into conversation history.

No request goes to Klique. No dependency installation is necessary for this baseline.

> **Screenshot placeholder:** The offline command, expected trace, and passing unittest summary. Do not include `.env` contents.

<!-- screenshot: images/vscode_agent_tests.png (Offline agent trace and passing unittest baseline.) -->

## 5D.3 Discover and Debug Tests in the IDE

1. Open **Testing** in the activity bar, then **Configure Python Tests** (or run **Python: Configure Tests**).
2. Choose **unittest**, `05_VSCode` as the tests directory, and `test_agent_lab.py` as the file pattern. If the wizard only offers patterns, choose `test*.py`; the other supplementary lab tests may also appear.
3. Refresh discovery, expand `AgentTests`, and run `test_json_valid_tools_and_history`.
4. Open `agent_lab.py`; click the gutter next to `result = run_tool(obj["tool"], obj["arguments"])` in the JSON branch.
5. In Testing, use **Debug Test** on that test, not ordinary **Run Test**.

When paused, inspect Variables and add Watch expressions:

```python
step
obj
history[-1]
tokens
sum(tokens.values())
row
```

Use **Step Into** to enter `run_tool`, inspect the selected function, then **Step Over** to execute it. At the first tool call, `obj["tool"]` is `lookup`. The result contains 8 GPUs at $2.50 per GPU-hour. Continue to the second pause: the expression is `8 * 2.5 * 3` and the result is `60.0`.

Use the Debug Console for read-only expressions. Do not invoke `client.complete()` from the console: in a live run that would make another metered request. Do not inspect or screenshot the live client's token or environment.

**What changed?** `history` contains the model's request, then the tool result as a user message in JSON mode. Native mode instead uses `role="tool"` and the matching `tool_call_id`. Debug `test_native_valid_tool` to compare the two paths.

> **Checkpoint:** You can explain why a tool result is added before the next model request.

See [VS Code Python testing](https://code.visualstudio.com/docs/python/testing) and [Python debugging](https://code.visualstudio.com/docs/python/debugging).

## 5D.4 Debug the Script Directly

Tests are the most precise entry point. To explore the entire demo, open `agent_lab.py` and use **Run and Debug → Python Debugger → Python File**. It runs the offline default without arguments.

For a repeatable launch, merge the following configuration into your repository-root `.vscode/launch.json` using the editor. Do not overwrite existing launch configurations. Create the file only if it does not exist.

```json
{
  "version": "0.2.0",
  "configurations": [
    {
      "name": "Agent lab: offline",
      "type": "debugpy",
      "request": "launch",
      "program": "${workspaceFolder}/05_VSCode/agent_lab.py",
      "cwd": "${workspaceFolder}",
      "console": "integratedTerminal",
      "justMyCode": true,
      "args": []
    }
  ]
}
```

Place a second breakpoint at `tokens[key] += count`. Watch `key`, `count`, `tokens`, and `step`. On step 2, the prompt total should move from 10 to 20, not stay at 10. A breakpoint pauses **before** executing its line.

Add a conditional breakpoint on that line with `step == 2 and key == "prompt"` to skip the first reply. Inspect the Call Stack to see `main → run_agent` or `test → run_agent`, depending on your launch path.

> **Screenshot placeholder:** Paused offline loop with `step`, `tokens`, `history`, and the Call Stack visible.

<!-- screenshot: images/vscode_agent_breakpoint.png (Offline agent paused with cumulative usage and message history.) -->

## 5D.5 Introduce a Bug Only in a Practice Copy

Leave the shipped module and tests passing. Create two **new** files from the repository root with this terminal command. Exclusive creation refuses to overwrite earlier practice work.

```bash
python - <<'PY'
from pathlib import Path
folder = Path("05_VSCode")
for source, destination in [
    ("agent_lab.py", "agent_lab_practice.py"),
    ("test_agent_lab.py", "test_agent_lab_practice.py"),
]:
    text = (folder / source).read_text(encoding="utf-8")
    if source.startswith("test_"):
        text = text.replace("from agent_lab import", "from agent_lab_practice import")
    with (folder / destination).open("x", encoding="utf-8") as output:
        output.write(text)
PY
```

Open `agent_lab_practice.py`. Find this line and deliberately remove the plus sign:

```python
# Correct:
tokens[key] += count
# Deliberately broken in the practice copy only:
tokens[key] = count
```

Run only the copied tests:

```bash
python -m unittest discover -s 05_VSCode -p test_agent_lab_practice.py -v
```

Expected: `test_json_valid_tools_and_history` fails because the tally is `{'prompt': 10, 'completion': 5}`, not `{'prompt': 30, 'completion': 15}`. Most other tests still pass: a single-response token-limit test cannot detect a multi-step accumulation bug.

In Testing, select `test*.py` if needed, refresh, and **Debug Test** on the failing test under `test_agent_lab_practice`. Set the conditional breakpoint in the **practice module**, on its assignment line, with `step == 2 and key == "prompt"`. Step over it and observe 10 replacing 10, instead of adding 10.

Repair the assignment to `+=`, save, and rerun the copied tests. All 20 should pass. Rerun the original tests too.

**Reflection:** Why would the agent still print a plausible final answer while undercounting usage? Why is a deterministic three-step fixture useful here?

> **Checkpoint:** You reproduced a failure, inspected its cause, repaired it, and verified both the practice and original suites. The repository's default agent was never intentionally broken.

## 5D.6 Test Stops, Not Just Answers

Read these tests and debug one from each category:

| Behavior | Test / expected result |
|---|---|
| Malformed JSON twice | `test_malformed_json_stops_after_two_responses`: `parse_error` |
| Invalid native arguments | `test_native_malformed_arguments_stop`: immediate `parse_error` |
| Unknown tool such as shell | `test_unknown_tool_returns_error_and_does_not_execute`: error returned to model, no shell execution |
| Rate-limit exhaustion | `test_rate_limit_exhaustion_stops_without_retry`: one call, `stopped_by_gateway` |
| Budget rejection | `test_budget_rejection_stops_without_retry`: one call, no retry |
| Unknown 429 | `test_unknown_429_is_not_assumed_to_be_rate_limit`: remains `ambiguous_limit` |
| Authentication / missing alias / timeout | `test_other_gateway_failures_do_not_change_mode_or_retry`: stop without fallback |
| Step cap | `test_step_cap`: one call, `max_steps` |
| Exact token threshold | `test_token_cap_at_boundary_before_tool_execution`: no tool or subsequent request |
| Final response exceeds threshold | `test_token_cap_overrides_final_answer`: `token_limit`, not `done` |
| Missing usage | `test_missing_usage_fails_closed`: `usage_unavailable` |

The rate-limit test does **not** exhaust a real quota. It simulates the server saying the limit is exhausted. Default retry allowance here is zero.

### What the limits do and do not guarantee

`max_steps=6` limits requests to six, with no automatic retry. `max_tokens=256` asks for at most 256 output tokens per response. `max_run_tokens=6000` counts reported input plus output usage after each response, before running tools or making another request.

The request that reaches the observed-token threshold has already happened. Input history grows, and server token accounting is authoritative; this client threshold can overshoot and is **not** a hard spending cap. Missing usage stops the run rather than silently counting zero. Wallet dollar budgets are configured and enforced at the gateway, separately. Do not turn enforcement off to make a test pass.

The tools are intentionally small and read-only. Native responses are limited to four tool calls each, arithmetic has size/range bounds, and only allowlisted functions run. These are teaching safeguards, not a production agent sandbox.

## 5D.7 Optional: One Bounded Live Run

Skip this section unless you have the governed endpoint and permissions from Sections 1–4. Review the shared setup in [5B](05b_Build_a_Governed_AI_Playground.ipynb) and keep credentials in the ignored `.env`, never the notebook, launch JSON, terminal command, or test fixture. Confirm the helper's configured gateway base URL ends in `/llm`, the intended profile is selected, and the first configured model alias exists in that profile. Do not print the token.

From the repository root, explicitly opt in:

```bash
python 05_VSCode/agent_lab.py --live --mode json
```

This creates `GatewayConfig.from_environment()` and `GatewayClient(config)`, chooses `config.models[0]`, and runs at most six requests. It does not probe another model, retry an error, or loop until a budget is exhausted. It can incur charges. Expected successful behavior is lookup, calculation, then an answer of $60, but the exact number of steps, usage, wording, and success depend on the model. A non-success stop exits with status 1; inspect its trace before deciding what to change.

JSON fallback is a prompt convention, not native function calling. Use native mode only after Section 4's successful capability check for **this exact endpoint/profile/model**:

```bash
python 05_VSCode/agent_lab.py --live --mode native
```

Choose **one** live command, not both by default. Native mode requires compatible model deployment settings, parser, and gateway behavior. A 401, 403, 404, budget refusal, or ambiguous 429 is not evidence that JSON fallback will fix the problem; the run stops without changing modes. A 200 with a normal answer is also not proof that native tools work. There is no silent probe or fallback in this module.

If a live run fails, keep the status and safe trace, check the profile/model configuration and administrator guidance, and do not repeatedly resubmit. A timeout may have reached the server and incurred usage already.

If authorized to view it, compare the run's recorded token counts with **AI Usage → Gateway**, filtered to your time range/model/user. Other traffic can change the dashboard totals. Do not assume rejected calls appear there.

> **Checkpoint:** Record whether the run was mocked or live, the selected mode, status, steps, and observed tokens. Do not label an unexecuted live test as verified.

## 5D.8 Save and Clean Up

1. Finish or stop the debugger. Save your code and record the bug and fix in your own lab notes.
2. Inspect `git status --short` and `git diff`. The shipped `agent_lab.py` and `test_agent_lab.py` should be unchanged by the practice exercise.
3. Keep practice files locally if useful, or remove only the two copies you created after confirming their names. Do not commit `.env`, secrets, live conversation history, or debug screenshots containing private data.
4. Keep only the intended debug/test settings if you created `.vscode` files. Merge rather than replace shared settings.
5. Stop your remote IDE using [Section 5's lifecycle steps](05_Running_VS_Code.ipynb) and verify the instance status. Closing the editor tab is not shutdown. This lab does not stop or modify a separate model deployment.

## Troubleshooting

| Symptom | Check |
|---|---|
| No tests discovered | Open the repository root, select the right Python, choose unittest and the correct folder/pattern, refresh discovery |
| Breakpoint hollow or never reached | Debug rather than Run; check interpreter and whether the test imports original or practice module |
| Practice test still passes after the edit | Save the file; confirm the copied test imports `agent_lab_practice`, not `agent_lab` |
| Practice file already exists | Inspect and reuse it; the creation command intentionally refuses to overwrite it |
| `ModuleNotFoundError` in an ad-hoc command | Use the documented `unittest discover -s 05_VSCode` or script path from the root |
| Debugger extension unavailable | Use terminal tests now and ask for an approved IDE image/extension setup before breakpoint work |
| `parse_error` live | Inspect safe response/history locally; malformed/truncated output is not a tool invocation; do not blindly raise limits |
| `usage_unavailable` | The provider did not return usable usage; stop and check its API behavior before trusting accounting |
| `stopped_by_gateway` | Keep the status/kind; check credentials, profile/alias, administrator-observed policy, and timeout outcome; no automatic retries |
| `token_limit` on a final answer | The post-response tally takes priority even over a final answer; observed usage may exceed the threshold |
| Native tool error | Check Section 4's deployment/parser prerequisites; do not interpret unrelated HTTP errors as unsupported tools |

## Wrap-Up

You now have a tested, inspectable agent loop and a repeatable way to debug failures without first spending tokens. The key lesson is that a plausible answer does not prove correct control flow, tool handling, or accounting.

**Back:** [VS Code introduction](05_Running_VS_Code.ipynb) · [Part 4 agent](../04_Agents/04_Agents_on_a_Governed_Endpoint.ipynb)  
**Index:** [Repository README](../README.md)

**Related references:** [VS Code Python testing](https://code.visualstudio.com/docs/python/testing) · [Python debugging](https://code.visualstudio.com/docs/python/debugging) · [Klique vLLM tool settings](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment) · [Gateway rate limits and wallet responses](https://klique.ai/docs/latest/docs/references/api/routers) · [AI Usage](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)